In [1]:
from nrem_analysis.constant import PROCESSED_DIR, MOUSE_IDS_TTX, FIGURES_DIR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import pynapple as nap
import cmap

norm = plt.Normalize(0, 2*np.pi)

In [ ]:
bins = 360
output = FIGURES_DIR / "tuning_curves"
output.mkdir(exist_ok=True)
UPDATE = True

for mouse_id in MOUSE_IDS_TTX:
    hd_units = nap.load_file(PROCESSED_DIR / 'ttx' / mouse_id / 'hd_units.npz')
    head_direction = nap.load_file(PROCESSED_DIR / 'ttx' / mouse_id / 'head_direction.npz')

    tcs = nap.compute_tuning_curves(
        data=hd_units,
        bins=bins,
        features=np.deg2rad(head_direction),
        range=(0, 2*np.pi),
        feature_names=['head_direction']
    )

    tcs.values = scipy.ndimage.gaussian_filter1d(tcs.values, sigma=3, axis=1, mode='wrap')
    pref_angles = tcs.idxmax('head_direction').values
    order = np.argsort(pref_angles)

    if UPDATE:
        hd_units.set_info(tcs.idxmax('head_direction').to_dataframe())
        hd_units.save(PROCESSED_DIR / 'ttx' / mouse_id / 'hd_units.npz')

    ncols = 16
    nrows = len(hd_units) // ncols + 1
    figsize = (ncols*4, nrows*4)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, subplot_kw={'projection': 'polar', 'theta_direction': -1}, constrained_layout=True)
    fig.suptitle(mouse_id, fontsize=26)
    axes = axes.flatten()

    for ax in axes:
        ax.set_visible(False)
    
    cyclic_cmap = cmap.Colormap('cmocean:phase').to_mpl()
    cyclic_colors = cyclic_cmap(norm(pref_angles))

    binned_angles = tcs.coords['head_direction']
    for i, idx in enumerate(order):
        tuning_curve = tcs.values[idx]
        axes[i].plot(binned_angles, tuning_curve, linewidth=1.5, color='k')
        axes[i].fill(binned_angles, tuning_curve, color=cyclic_colors[idx], alpha=0.5)
        axes[i].axvline(pref_angles[idx], c='k', linewidth=1.2, alpha=0.8)

        axes[i].set_visible(True)
        axes[i].set_yticks([])
        axes[i].set_xticks([0, np.pi/2, np.pi, 3*np.pi/2])
        axes[i].set_xticklabels([])
        axes[i].set_title(f'{hd_units.index[idx].item()}', fontsize=22)
    
    plt.savefig(output/f"{mouse_id}.png", dpi=320)
    plt.close(fig)